In [1]:
import os
import requests
import json
from dotenv import load_dotenv
import requests_cache
import re, time

In [22]:
# load .env file, get API key
#load_dotenv()
#API_KEY = os.getenv("api_key")

# make sure API key is available
#if not API_KEY:
    #raise ValueError("no API key found")

# set up a cached session
session = requests_cache.CachedSession('output/steam_api_cache', expire_after=86400)

In [24]:
# sample output
def fetch_steam_app_data(app_id: int) -> dict:
    """
    Fetch detailed information about a Steam application using the Storefront API.
    """
    
    # define the target API endpoint
    url = "https://store.steampowered.com/api/appdetails"
    
    # construct query parameters for the GET request
    params = {
        "appids": app_id,
        "cc": "us",      # currency: USD
        "l": "english"   # language: English
    }
    
    try:

        # use cached session to sent request, timeout after 10 seconds
        response = session.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        # parse the JSON response into a json
        raw_data = response.json()
        
        # check if the response contains valid data for the given app_id
        app_id_str = str(app_id)
        if app_id_str in raw_data and raw_data[app_id_str].get("success"):
            return raw_data[app_id_str]["data"]
        else:
            print(f"No valid data found for AppId:{app_id_str}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"HTTP Request failed: {e}")
        return None


In [26]:
sample_output = fetch_steam_app_data(1091500)

with open("output/sample_app_data.json", "w") as f:
    json.dump(sample_output, f, indent=4)

In [ ]:
# 1) 从 Steam 商店搜索页抓一批 appid
#    filter="topsellers" 可换成 "newandtrending" / "specials"
def get_appids(target=2000, pages=120, flt="topsellers"):
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0"}) 

    appids, seen = [], set()
    for p in range(1, pages + 1):
        html = s.get(
            "https://store.steampowered.com/search/",
            params={"filter": flt, "page": p},
            timeout=30
        ).text

        # 搜索页里每个游戏条目都有 data-ds-appid="1091500"
        # 有时会是 "123,456"（例如包含多个 appid），所以后面要 split(",")
        for x in re.findall(r'data-ds-appid="([\d,]+)"', html):
            for a in x.split(","):
                a = int(a)
                if a not in seen:
                    seen.add(a)
                    appids.append(a)
                    if len(appids) >= target:
                        return appids

        # 翻页限速
        time.sleep(0.5 + random.random() * 0.3)

    return appids

# 2) 429 限流自动重试：指数退避（1.2s, 2.4s, 4.8s...）+ 随机抖动
def safe_fetch(appid, tries=7, base=1.2):
    for i in range(tries):
        try:
            return fetch_steam_app_data(appid)
        except Exception as e:
            # 只对 429 处理，其它错误就跳过该游戏
            if "429" not in str(e):
                return None

            wait = base * (2 ** i) + random.random()
            print(f"[429] appid={appid} wait={wait:.1f}s (try {i+1}/{tries})")
            time.sleep(wait)
    return None


# 3) 主循环：抓 1000 个游戏，输出 JSONL（每行一个 sample.json 结构）
TARGET = 1000
OUT = "steam_1000.jsonl"

appids = get_appids(target=TARGET * 2, pages=120, flt="topsellers")
random.shuffle(appids)  # 打乱顺序，避免全部都是榜单前排类型

n = 0
with open(OUT, "w", encoding="utf-8") as f:
    for appid in appids:
        d = safe_fetch(appid)

        # 只保留真正的游戏（过滤 demo / dlc / application）
        if d and (d.get("type") or "").lower() == "game":
            f.write(json.dumps(d, ensure_ascii=False) + "\n") 
            n += 1
            if n >= TARGET:
                break

        time.sleep(1.0 + random.random() * 0.5)

print(f"Done. saved={n} -> {OUT}")

HTTP Request failed: 502 Server Error: Bad Gateway for url: https://store.steampowered.com/api/appdetails?appids=3228590&cc=us&l=english
HTTP Request failed: 502 Server Error: Bad Gateway for url: https://store.steampowered.com/api/appdetails?appids=2745870&cc=us&l=english
HTTP Request failed: 502 Server Error: Bad Gateway for url: https://store.steampowered.com/api/appdetails?appids=3315510&cc=us&l=english
HTTP Request failed: 502 Server Error: Bad Gateway for url: https://store.steampowered.com/api/appdetails?appids=3116890&cc=us&l=english
HTTP Request failed: 502 Server Error: Bad Gateway for url: https://store.steampowered.com/api/appdetails?appids=2948190&cc=us&l=english
HTTP Request failed: 502 Server Error: Bad Gateway for url: https://store.steampowered.com/api/appdetails?appids=2973290&cc=us&l=english
Done. saved=1000 -> steam_1000.jsonl
